# 1. Load Metadata and Select References


In [ ]:
from pathlib import Path
import pandas as pd
import re

Root = Path.cwd()
if not (Root / "Formulation").exists():
    Root = Root.parent

CsvPath = Root / "Formulation" / "Literature" / "SImilar40" / "selected_scores_20260209_164223.csv"
BibPath = Root / "Formulation" / "Literature" / "SImilar40" / "knapsack_20260209_165848.bib"

DataFrame = pd.read_csv(CsvPath, usecols=["title", "abstract"], encoding="utf-8")
BibText = BibPath.read_text(encoding="utf-8")

KeyMatches = re.findall(r"@\w+\{([^,]+),", BibText)
AvailableKeys = set(KeyMatches)

PreferredKeys = [
    "büyüktahtakın_scenario_2023",
    "bos_distributionally_2024",
    "ding_balancing_2022",
    "chen_sample_2022",
    "larsen_fast_2024",
    "rezaeian_assignment_2024",
]

SelectedKeys = [Key for Key in PreferredKeys if Key in AvailableKeys]
SelectedTitles = DataFrame[DataFrame["title"].str.contains("knapsack|assignment|stochastic", case=False, na=False)]

SelectedKeys, SelectedTitles.head(5)


# 2. Update `Formulation.tex` from `Formulation_SJ.tex`


In [ ]:
from pathlib import Path

Root = Path.cwd()
if not (Root / "Formulation").exists():
    Root = Root.parent

FormulationPath = Root / "Formulation" / "Formulation.tex"
SourcePath = Root / "Collab" / "Formulation_SJ.tex"

FormulationText = FormulationPath.read_text(encoding="utf-8")
SourceText = SourcePath.read_text(encoding="utf-8")

if "Stochastic Multiple Knapsack Extension" in FormulationText:
    "Formulation.tex already includes the updated knapsack formulation."
else:
    MarkerStart = SourceText.find("\\section*{Problem Description}")
    MarkerEnd = SourceText.find("\\end{document}")
    Snippet = SourceText[MarkerStart:MarkerEnd].strip()
    "Formulation update required. Snippet extracted:", Snippet[:500]


# 3. Update `SLM.tex` with New Content and Citations


In [ ]:
from pathlib import Path

Root = Path.cwd()
if not (Root / "Presentation").exists():
    Root = Root.parent

SlidesPath = Root / "Presentation" / "SLM.tex"
SlidesText = SlidesPath.read_text(encoding="utf-8")

if "mkp_structure.png" in SlidesText and "Stochastic Multiple Knapsack" in SlidesText:
    "SLM.tex already includes updated slides and citations."
else:
    "SLM.tex update required."


# 4. Generate Stochastic Knapsack Figures


In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

Root = Path.cwd()
if not (Root / "Presentation").exists():
    Root = Root.parent

OutputDir = Root / "Presentation"
OutputDir.mkdir(parents=True, exist_ok=True)

RandomState = np.random.default_rng(42)
AgentCount = 3
TaskCount = 8

Agents = [f"Agent {Index + 1}" for Index in range(AgentCount)]
Tasks = [f"Task {Index + 1}" for Index in range(TaskCount)]
Capacities = RandomState.uniform(5.5, 7.5, size=AgentCount)
Sizes = RandomState.uniform(0.6, 2.0, size=TaskCount)
Returns = RandomState.uniform(2.0, 8.0, size=TaskCount)
Assignments = RandomState.integers(0, AgentCount, size=TaskCount)

AgentY = np.linspace(0.8, 0.2, AgentCount)
TaskY = np.linspace(0.9, 0.1, TaskCount)

fig, ax = plt.subplots(figsize=(9.0, 5.0))
for AgentIndex, AgentName in enumerate(Agents):
    ax.scatter(0.8, AgentY[AgentIndex], s=120, color="#1F4E5F")
    ax.text(0.84, AgentY[AgentIndex], f"{AgentName}\nC={Capacities[AgentIndex]:.1f}",
            va="center", fontsize=9)

for TaskIndex, TaskName in enumerate(Tasks):
    ax.scatter(0.2, TaskY[TaskIndex], s=90, color="#E09F3E")
    ax.text(0.16, TaskY[TaskIndex], TaskName, va="center", ha="right", fontsize=9)

for TaskIndex in range(TaskCount):
    AgentIndex = Assignments[TaskIndex]
    Width = 0.5 + Sizes[TaskIndex]
    ColorValue = Returns[TaskIndex]
    EdgeColor = plt.cm.viridis((ColorValue - Returns.min()) / (Returns.max() - Returns.min() + 1e-6))
    ax.plot([0.25, 0.75], [TaskY[TaskIndex], AgentY[AgentIndex]],
            color=EdgeColor, linewidth=Width, alpha=0.8)

ax.set_title("Multiple knapsack assignment graph")
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0.0, 1.0)
ax.axis("off")
fig.tight_layout()
fig.savefig(OutputDir / "mkp_structure.pdf")
plt.close(fig)

fig, ax = plt.subplots(figsize=(9.0, 5.0))
for AgentIndex, AgentName in enumerate(Agents):
    ax.scatter(0.8, AgentY[AgentIndex], s=120, color="#1F4E5F")
    ax.text(0.84, AgentY[AgentIndex], AgentName, va="center", fontsize=9)

for TaskIndex, TaskName in enumerate(Tasks):
    ax.scatter(0.2, TaskY[TaskIndex], s=90, color="#E09F3E")
    ax.text(0.16, TaskY[TaskIndex], TaskName, va="center", ha="right", fontsize=9)

for TaskIndex in range(TaskCount):
    AgentIndex = Assignments[TaskIndex]
    MeanSize = Sizes[TaskIndex]
    MeanReturn = Returns[TaskIndex]
    SizeSpread = RandomState.uniform(0.2, 0.6)
    ReturnSpread = RandomState.uniform(0.4, 1.2)
    BaseColor = plt.cm.magma((MeanReturn - Returns.min()) / (Returns.max() - Returns.min() + 1e-6))
    for SampleIndex in range(4):
        SampleSize = MeanSize + RandomState.normal(0.0, SizeSpread * 0.25)
        Alpha = 0.25 + 0.15 * SampleIndex
        ax.plot([0.25, 0.75], [TaskY[TaskIndex], AgentY[AgentIndex]],
                color=BaseColor, linewidth=0.8 + SampleSize, alpha=Alpha)
    ax.text(0.5, (TaskY[TaskIndex] + AgentY[AgentIndex]) / 2.0,
            f"R={MeanReturn:.1f}±{ReturnSpread:.1f}", fontsize=7,
            ha="center", va="center", color="#0B1D26")

ax.set_title("Stochastic sizes and returns on assignment edges")
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0.0, 1.0)
ax.axis("off")
fig.tight_layout()
fig.savefig(OutputDir / "mkp_stochastic.pdf")
plt.close(fig)


# 5. Compile and Clean Beamer PDF


In [ ]:
from pathlib import Path
import subprocess

Root = Path.cwd()
if not (Root / "Presentation").exists():
    Root = Root.parent

PresentationDir = Root / "Presentation"

BuildCommand = ["latexmk", "-xelatex", "SLM.tex"]
CleanCommand = ["latexmk", "-c"]

BuildResult = subprocess.run(BuildCommand, cwd=PresentationDir, capture_output=True, text=True)
CleanResult = subprocess.run(CleanCommand, cwd=PresentationDir, capture_output=True, text=True)

BuildResult.returncode, CleanResult.returncode
